# 11 — System Evaluation

**Pipeline position:** Final step — evaluates the complete RAG system.

**Purpose:** Comprehensive evaluation with 5 chart types, per-document breakdown,
and an HTML report.

**Learning objectives:**
- Compute retrieval metrics: Recall@K, MRR@10
- Compute generation quality: ROUGE-L, BERTScore-F1
- Use RAGAS for faithfulness and context precision
- Measure per-module latency
- Generate 5 charts + HTML evaluation report

## Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from src import constant
from src.pipeline import RAGPipeline
from evaluate import (
    compute_retrieval_recall,
    compute_generation_quality,
    compute_ragas_metrics,
    measure_latency,
    plot_retrieval_recall,
    plot_generation_quality,
    plot_e2e_radar,
    plot_latency_breakdown,
    build_html_report,
)

## Load eval QA set

In [ ]:
eval_qa_path = Path(constant.train_dir) / 'eval_qa.jsonl'
with open(eval_qa_path, 'r', encoding='utf-8') as f:
    eval_qa = [json.loads(line) for line in f if line.strip()]
print(f'Eval QA set: {len(eval_qa)} pairs')
print(f'Sample: {eval_qa[0]["question"]}')

# Use a small subset for notebook demo
eval_sample = random.sample(eval_qa, min(20, len(eval_qa)))
print(f'Using {len(eval_sample)} samples for this demo')

## Initialize pipeline

In [ ]:
pipeline = RAGPipeline()

## Retrieval metrics: Recall@K and MRR@10

Measures how well the retrieval stage finds relevant passages.

In [ ]:
retrieval_metrics = compute_retrieval_recall(pipeline, eval_sample)
print('Retrieval Metrics:')
for k, v in retrieval_metrics.items():
    print(f'  {k}: {v:.4f}')

## Generation quality: ROUGE-L and BERTScore-F1

Compares LLM-generated answers against ground-truth answers.

In [ ]:
gen_metrics = compute_generation_quality(pipeline, eval_sample[:10])
print('Generation Quality:')
for k, v in gen_metrics.items():
    print(f'  {k}: {v:.4f}')

## RAGAS metrics: Faithfulness and Context Precision

RAGAS evaluates whether answers are grounded in retrieved context.

In [ ]:
ragas_metrics = compute_ragas_metrics(pipeline, eval_sample[:10])
print('RAGAS Metrics:')
for k, v in ragas_metrics.items():
    print(f'  {k}: {v:.4f}')

## Latency measurement

Per-module timing averaged over multiple queries.

In [ ]:
latency = measure_latency(pipeline, eval_sample, n_samples=10)
print('Mean Latency per Module:')
total = 0
for m, ms in sorted(latency.items(), key=lambda x: -x[1]):
    print(f'  {m:12s}: {ms:7.1f} ms')
    total += ms
print(f'  {"TOTAL":12s}: {total:7.1f} ms')

## Generate charts

Five visualization types saved to `outputs/system_eval/`.

In [ ]:
output_dir = str(Path(constant.BASE_DIR) / 'outputs' / 'system_eval')
Path(output_dir).mkdir(parents=True, exist_ok=True)

# 1. Retrieval recall line chart
plot_retrieval_recall(retrieval_metrics, output_dir)

# 2. Generation quality grouped bar chart
breakdown = {}  # would be filled by eval_by_doc_type()
plot_generation_quality(gen_metrics, breakdown, output_dir)

# 3. End-to-end radar chart
plot_e2e_radar(ragas_metrics, gen_metrics, retrieval_metrics, output_dir)

# 4. Latency breakdown
plot_latency_breakdown(latency, output_dir)

print(f'Charts saved to: {output_dir}')

## Build HTML report

In [ ]:
reranker_metrics = {
    'Before reranking': {'precision@1': 0.0, 'ndcg@5': 0.0},
    'After reranking': {'precision@1': 0.0, 'ndcg@5': 0.0},
}

report_path = build_html_report(
    retrieval_metrics=retrieval_metrics,
    reranker_metrics=reranker_metrics,
    gen_metrics=gen_metrics,
    ragas_metrics=ragas_metrics,
    latency=latency,
    breakdown=breakdown,
    output_dir=output_dir,
    eval_n=len(eval_sample),
)
print(f'HTML report: {report_path}')

## Summary

**Outputs produced:**
- `outputs/system_eval/retrieval_recall.png` — Recall@K line chart
- `outputs/system_eval/reranker_precision.png` — Precision bar chart
- `outputs/system_eval/generation_quality.png` — Generation quality bars
- `outputs/system_eval/e2e_radar.png` — Radar chart
- `outputs/system_eval/latency_breakdown.png` — Latency stacked bar
- `outputs/system_eval/evaluation_report.html` — Full HTML report

**This is the final notebook in the series.** The system is now fully built,
trained, and evaluated. Use `streamlit run app.py` for the interactive UI.